In [3]:
import shap
from sklearn.model_selection import train_test_split

In [4]:
import torch
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from transformers import pipeline, set_seed
from collections import defaultdict

/Users/pranitgunjal/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [40]:
sentence_transformer = SentenceTransformer('all-mpnet-base-v2')

In [5]:
df = pd.read_csv('../data/initial_datasets/reddit/reddit_train.csv')

In [42]:
df

,text,label
0,That game hurt.,-1
1,Man I love reddit.,1
2,Right? Considering it’s such an important docu...,1
3,"He isn't as big, but he's still quite popular....",-1
4,That's crazy; I went to a super [RELIGION] hig...,1
...,...,...
40696,Oh man is this true. Coffee has a seriously ne...,-1
40697,"You’re good, no worries",1
40698,"one's a rapist, and the other's a stingy yank ...",-1
40699,This is great! Can anyone make a request with ...,1


In [43]:
cleaned_df = df[~(df['action'] == 'UNCLEAR')]

KeyError: 'action'

In [ ]:
cleaned_df['action'] = cleaned_df['action'].replace({"NOT-PROSOCIAL": 0, "PROSOCIAL": 1})

/var/folders/lv/pnwq6bmj4tq68bsvy__37qyh0000gn/T/ipykernel_37843/3109795256.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  cleaned_df['action'] = cleaned_df['action'].replace({"NOT-PROSOCIAL": 0, "PROSOCIAL": 1})
/var/folders/lv/pnwq6bmj4tq68bsvy__37qyh0000gn/T/ipykernel_37843/3109795256.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleaned_df['action'] = cleaned_df['action'].replace({"NOT-PROSOCIAL": 0, "PROSOCIAL": 1})


In [6]:
cleaned_df = df.sample(n=1000)
cleaned_df

,text,label
5268,That’s awesome!,1
27832,SOME newborns are ugly. I did think my little ...,-1
17100,Dumpster fire. I cant quit you Edmonton!!,-1
27198,i got a bump and a bald spot. i feel dumb <3,-1
2092,"I'll handle what ever comes my way, I adore ou...",-1
...,...,...
24550,#I am relaxed,1
30785,It is a racist nickname because you're literal...,-1
21112,"First [NAME], now [NAME]. Stop the conks. Plea...",-1
33580,Really curious to know how you know what peopl...,1


In [ ]:
labels = cleaned_df['label'].to_list()

X = sentence_transformer.encode(cleaned_df['text'].to_list(), convert_to_numpy=True)

In [14]:
classifier = pipeline(
    "text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    return_all_scores=True,
    verbose=False
)

Device set to use mps:0
/Users/pranitgunjal/Library/Python/3.9/lib/python/site-packages/transformers/pipelines/text_classification.py:106: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [ ]:
def f(texts):
    preds = classifier(texts)
    return [[p["score"]] if p["label"] == "POSITIVE" else [1 - p["score"]] for p in preds]


In [ ]:
explainer = shap.Explainer(f, shap.maskers.Text(tokenizer=classifier.tokenizer))

In [7]:
import shap
from transformers import pipeline
import numpy as np

# Load sentiment analysis pipeline
pipe = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

# Wrapper function for SHAP
def model_predict(texts):
    outputs = pipe(texts)
    return np.array([[o['score']] if o['label'] == 'POSITIVE' else [1 - o['score']] for o in outputs])

# Set up SHAP masker and explainer
masker = shap.maskers.Text(tokenizer=pipe.tokenizer)
explainer = shap.Explainer(model_predict, masker)

# Now explain the prediction
input_text = "You're an awesome person."
shap_values = explainer([input_text])  # input as list of strings

# Visualize
shap.plots.text(shap_values[0])


Device set to use mps:0


ValueError: text input must be of type `str` (single example), `List[str]` (batch or single pretokenized example) or `List[List[str]]` (batch of pretokenized examples).

In [11]:
import shap
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Load model and tokenizer directly (not pipeline!)
model_name = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Set model to eval mode
model.eval()

# Create SHAP-compatible prediction function
def f(texts):
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.nn.functional.softmax(logits, dim=1)
    return probs.numpy()

# Set up the SHAP text masker and explainer
masker = shap.maskers.Text(tokenizer)
explainer = shap.Explainer(f, masker)

# Text to explain
text = str("You're an awesome person.")
print(type(text))
# Get SHAP values
shap_values = explainer(text)

# Visualize
shap.plots.text(shap_values[0])


<class 'str'>


ValueError: text input must be of type `str` (single example), `List[str]` (batch or single pretokenized example) or `List[List[str]]` (batch of pretokenized examples).

In [ ]:
shap_values = explainer(["Your an awesome person."])
shap.plots.text(shap_values[0])

ValueError: text input must be of type `str` (single example), `List[str]` (batch or single pretokenized example) or `List[List[str]]` (batch of pretokenized examples).

In [15]:
masker = shap.maskers.Text(classifier.tokenizer)
explainer = shap.Explainer(classifier, masker, verbose=False)

In [16]:
shap_values = explainer(["Your a dumbass"])

In [17]:
shap.plots.text(shap_values[0])

In [ ]:
chats = cleaned_df['text'].sample(n=100).to_list()
shap_values = explainer(chats)

PartitionExplainer explainer:   5%|▌         | 5/100 [00:14<03:25,  2.16s/it]

  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer:   6%|▌         | 6/100 [00:19<05:22,  3.43s/it]

  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer:   7%|▋         | 7/100 [00:24<06:15,  4.04s/it]

...

PartitionExplainer explainer:  95%|█████████▌| 95/100 [05:05<00:14,  2.84s/it]

  0%|          | 0/240 [00:00<?, ?it/s]

PartitionExplainer explainer: 100%|██████████| 100/100 [05:27<00:00,  3.85s/it]

  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer: 101it [05:32,  3.40s/it]                         


In [ ]:
prosocial_importance = defaultdict(float)
not_prosocial_importance = defaultdict(float)

In [ ]:
for explanation in shap_values:
    for token, value in zip(explanation.data, explanation.values):
        prosocial_importance[token] += abs(value[0])
        not_prosocial_importance[token] += abs(value[1])

In [ ]:
prosocial_top = sorted(prosocial_importance.items(), key=lambda x: x[1], reverse=True)
not_prosocial_top = sorted(not_prosocial_importance.items(), key=lambda x: x[1], reverse=True)

In [ ]:
for token, score in prosocial_top[:10]:
    print(f"{token}: {score:.4f}")

.: 1.8760
. : 1.6809
not : 1.6022
good : 1.5848
awesome: 1.5322
I : 1.3429
Good : 1.3063
a : 1.2847
cr: 1.2275
t : 1.1827


In [ ]:
for token, score in not_prosocial_top[:10]:
    print(f"{token}: {score:.4f}")

.: 1.8760
. : 1.6809
not : 1.6022
good : 1.5848
awesome: 1.5322
I : 1.3429
Good : 1.3063
a : 1.2847
cr: 1.2275
t : 1.1827


In [ ]:
token_importance_scalar = {
    token: np.sum(np.abs(val)) for token, val in token_importance.items()
}

NameError: name 'token_importance' is not defined

In [ ]:
top_tokens = sorted(token_importance_scalar.items(), key=lambda x: x[1], reverse=True)

In [ ]:
for token, importance in top_tokens[:20]:
    print(f"{token}: {importance:.4f}")

NameError: name 'top_tokens' is not defined